# v26 Full — next-session operational runner

No training. No generation. Freeze MC from v25/v24.1 gold-strict; experiment only on YNU post-processing. Outputs use explicit suffixes: standard / a / b / full.

In [ ]:

# ==================================================================
# v26 COMMON -- helpers
# ==================================================================
import os, re, json, glob, math, random
from pathlib import Path
from collections import Counter, defaultdict

VERSION = "v26"
BASE_INPUT_DIRS = [
    "/kaggle/input/datasets/yixuanisthebest/v2333333",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline",
    "/kaggle/working",
]
OUT_DIR = Path("/kaggle/working/v26_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["A", "B", "C", "D", "Yes", "No", "Unknown"]
MC_LABELS = {"A", "B", "C", "D"}
YNU_LABELS = {"Yes", "No", "Unknown"}


def find_one(patterns, required=True, name="file"):
    hits = []
    for pat in patterns:
        hits.extend(glob.glob(pat, recursive=True))
    hits = sorted(set(hits), key=lambda p: ("/kaggle/working" not in p, len(p), p))
    print(f"{name} candidates:", hits[:10])
    if not hits:
        if required:
            raise FileNotFoundError(f"Could not find {name}; patterns={patterns}")
        return None
    return hits[0]

CAND_PATH = find_one([
    "/kaggle/input/**/v23_val_candidates.json",
    "/kaggle/working/**/v23_val_candidates.json",
], required=True, name="v23_val_candidates")

DEBUG_PATH = find_one([
    "/kaggle/input/**/v24_1_optionwise_mc_debug*.json",
    "/kaggle/working/**/v24_1_optionwise_mc_debug*.json",
], required=True, name="v24_1_optionwise_mc_debug")

print("CAND_PATH =", CAND_PATH)
print("DEBUG_PATH=", DEBUG_PATH)

with open(CAND_PATH, "r", encoding="utf-8") as f:
    RESULTS = json.load(f)
with open(DEBUG_PATH, "r", encoding="utf-8") as f:
    MC_DEBUG = json.load(f)
print("Loaded results:", len(RESULTS))
print("Loaded MC debug rows:", len(MC_DEBUG))


def norm_answer(x):
    if x is None:
        return "UNPARSEABLE"
    s = str(x).strip()
    if not s:
        return "UNPARSEABLE"
    t = s.upper()
    if t in {"A","B","C","D"}: return t
    if t in {"YES","TRUE"}: return "Yes"
    if t in {"NO","FALSE"}: return "No"
    if t in {"UNKNOWN","UNK","CANNOT DETERMINE","NOT ENOUGH INFORMATION"}: return "Unknown"
    # catch final-answer-like strings
    m = re.search(r"\b(A|B|C|D|Yes|No|Unknown)\b", s, flags=re.I)
    if m:
        return norm_answer(m.group(1))
    return "UNPARSEABLE"


def get_gold(r):
    return norm_answer(r.get("gold") or r.get("answer") or r.get("label") or r.get("target"))


def is_mc_record(r):
    # VAL gold-strict: use gold only to avoid over-detecting YNU as MC
    return get_gold(r) in MC_LABELS


def candidate_answer(c):
    return norm_answer(c.get("answer") or c.get("pred") or c.get("final_answer"))


def candidate_text(c):
    parts = []
    for k in ["text", "raw", "output", "explanation", "reasoning"]:
        v = c.get(k)
        if isinstance(v, str): parts.append(v)
    return "\n".join(parts)


def supporting_count(c):
    sp = c.get("supporting_premises") or c.get("support") or c.get("cited_premises") or []
    if isinstance(sp, str):
        nums = re.findall(r"\d+", sp)
        return len(nums)
    if isinstance(sp, (list, tuple)):
        return len(sp)
    return 0


def majority_pred(r):
    ans = [candidate_answer(c) for c in r.get("candidates", [])]
    ans = [a for a in ans if a in LABELS]
    if not ans:
        return "UNPARSEABLE"
    cnt = Counter(ans)
    return cnt.most_common(1)[0][0]


def first_pred(r):
    cs = r.get("candidates", [])
    if not cs: return "UNPARSEABLE"
    return candidate_answer(cs[0])


def oracle_pred(r):
    gold = get_gold(r)
    for c in r.get("candidates", []):
        if candidate_answer(c) == gold:
            return gold
    return majority_pred(r)


def score_candidate(c, params):
    ans = candidate_answer(c)
    text = candidate_text(c)
    n_supp = supporting_count(c)
    score = 0.0
    score += params.get("w_cite", 0.0) * (1.0 if n_supp > 0 else 0.0)
    score += params.get("w_short_supp", 0.0) * (1.0 if 1 <= n_supp <= 5 else 0.0)
    if n_supp > 8:
        score -= params.get("p_long_supp", 0.0)
    tl = text.lower()
    if any(x in tl for x in ["contradict", "not supported", "cannot infer", "not entailed"]):
        score -= params.get("p_contra_text", 0.0)
    if ans == "Yes": score += params.get("yes_boost", 0.0)
    if ans == "No": score -= params.get("no_penalty", 0.0)
    if ans == "Unknown": score -= params.get("unk_penalty", 0.0)
    return score


def rerank_ynu(r, params):
    cs = r.get("candidates", [])
    if not cs:
        return "UNPARSEABLE"
    answers = [candidate_answer(c) for c in cs]
    vote = Counter([a for a in answers if a in YNU_LABELS])
    total = max(1, sum(vote.values()))
    best = None
    best_score = -1e9
    for c in cs:
        ans = candidate_answer(c)
        if ans not in YNU_LABELS:
            continue
        s = params.get("w_vote", 1.0) * (vote[ans] / total) + score_candidate(c, params)
        if s > best_score:
            best_score = s
            best = ans
    return best or majority_pred(r)


def explicit_answer_from_text(text):
    """Extract explicit natural-language answer signals from reasoning text.
    This is intentionally conservative in standard; aggressive notebooks add more patterns.
    """
    if not isinstance(text, str) or not text.strip():
        return None
    # Prefer final answer if present
    ms = list(re.finditer(r"Final\s*Answer\s*[:\-]?\s*(Yes|No|Unknown|A|B|C|D)\b", text, flags=re.I))
    if ms:
        return norm_answer(ms[-1].group(1))
    ms = list(re.finditer(r"\b(answer|therefore|thus|so)\b[^\n\.]{0,80}\b(Yes|No|Unknown)\b", text, flags=re.I))
    if ms:
        return norm_answer(ms[-1].group(2))
    return None


def repair_ynu_by_text(r, pred, mode="standard"):
    if pred not in YNU_LABELS and pred != "UNPARSEABLE":
        return pred
    # choose strongest candidate text with explicit final answer or semantic conclusion
    cs = r.get("candidates", [])
    signals = []
    for c in cs:
        ans = candidate_answer(c)
        text = candidate_text(c)
        sig = explicit_answer_from_text(text)
        if sig in YNU_LABELS:
            signals.append(sig)
    if not signals:
        return pred
    cnt = Counter(signals)
    top, n = cnt.most_common(1)[0]
    # Standard: only repair unparseable, or when explicit signal has at least 2 votes
    if mode == "standard":
        if pred == "UNPARSEABLE" or n >= 2:
            return top
        return pred
    # Conservative: only repair UNPARSEABLE
    if mode == "conservative":
        return top if pred == "UNPARSEABLE" else pred
    # Aggressive: repair any pred if explicit signals disagree and top count >=1
    if mode == "aggressive":
        return top
    return pred


def mc_pred_from_debug(idx):
    d = MC_DEBUG.get(str(idx)) if isinstance(MC_DEBUG, dict) else None
    if d is None:
        return None
    support = d.get("support") or {}
    true_opts = [o for o in ["A","B","C","D"] if bool(support.get(o))]
    if len(true_opts) == 1:
        return true_opts[0]
    if len(true_opts) > 1:
        # Deterministic tie: prefer option with longest evidence/raw yes text; fallback A/B/C/D order.
        raw = d.get("raw") or {}
        def raw_len(o): return len(str(raw.get(o, "")))
        return sorted(true_opts, key=lambda o: (-raw_len(o), o))[0]
    return "Unknown"


def metric_from_preds(preds, golds, name="metric"):
    n = len(golds)
    acc = sum(p == g for p,g in zip(preds,golds)) / max(1,n)
    per = {}
    f1s=[]; weighted=[]
    for lab in LABELS:
        tp = sum(p==lab and g==lab for p,g in zip(preds,golds))
        fp = sum(p==lab and g!=lab for p,g in zip(preds,golds))
        fn = sum(p!=lab and g==lab for p,g in zip(preds,golds))
        support = sum(g==lab for g in golds)
        pred_count = sum(p==lab for p in preds)
        prec = tp/(tp+fp) if tp+fp else 0.0
        rec = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*prec*rec/(prec+rec) if prec+rec else 0.0
        per[lab] = dict(precision=prec, recall=rec, f1=f1, support=support, pred_count=pred_count, tp=tp, fp=fp, fn=fn)
        f1s.append(f1); weighted.append(f1*support)
    mc_macro = sum(per[x]["f1"] for x in ["A","B","C","D"])/4
    ynu_macro = sum(per[x]["f1"] for x in ["Yes","No","Unknown"])/3
    return dict(name=name, n=n, acc=acc, macro_f1=sum(f1s)/len(f1s), weighted_f1=sum(weighted)/max(1,n), mc_macro_f1=mc_macro, ynu_macro_f1=ynu_macro, per_label=per)


def build_preds(params, repair_mode="standard"):
    rows=[]; preds=[]; golds=[]
    missing_debug = 0
    for i,r in enumerate(RESULTS):
        gold = get_gold(r)
        if is_mc_record(r):
            pred = mc_pred_from_debug(i)
            if pred is None:
                missing_debug += 1
                pred = majority_pred(r)
            source = "mc_optionwise_v241_goldstrict"
        else:
            pred = rerank_ynu(r, params)
            pred = repair_ynu_by_text(r, pred, mode=repair_mode)
            source = f"ynu_rerank_repair_{repair_mode}"
        preds.append(pred); golds.append(gold)
        rows.append({
            "idx": i,
            "gold": gold,
            "pred": pred,
            "is_mc": is_mc_record(r),
            "source": source,
            "question": str(r.get("question", ""))[:1000],
        })
    return preds, golds, rows, missing_debug


def save_json(path, obj):
    def conv(x):
        try:
            import numpy as np
            if isinstance(x, (np.integer,)): return int(x)
            if isinstance(x, (np.floating,)): return float(x)
            if isinstance(x, (np.bool_,)): return bool(x)
        except Exception:
            pass
        if isinstance(x, dict): return {str(k): conv(v) for k,v in x.items()}
        if isinstance(x, (list, tuple)): return [conv(v) for v in x]
        return x
    with open(path, "w", encoding="utf-8") as f:
        json.dump(conv(obj), f, ensure_ascii=False, indent=2)
    print("saved", path)

BASE_PARAMS = {
    "w_vote": 1.0,
    "w_cite": 0.3,
    "w_short_supp": 0.15,
    "yes_boost": 0.3,
    "no_penalty": 0.15,
    "unk_penalty": 0.35,
    "p_long_supp": 0.2,
    "p_contra_text": 0.4,
}

GOLDS = [get_gold(r) for r in RESULTS]
print("Gold dist:", Counter(GOLDS))
print("MC count:", sum(g in MC_LABELS for g in GOLDS))


In [ ]:

# ==================================================================
# v26 FULL -- operational next-session runner
# ==================================================================
# Loads v26_standard_summary if available, otherwise uses BASE_PARAMS.
# Always writes suffix _full outputs.

standard_path = Path("/kaggle/working/v26_outputs/v26_standard_summary.json")
if standard_path.exists():
    with open(standard_path, "r", encoding="utf-8") as f:
        std = json.load(f)
    params = std.get("best_params", BASE_PARAMS)
    mode = "standard"
    print("Loaded params from", standard_path)
else:
    params = BASE_PARAMS
    mode = "standard"
    print("No v26_standard_summary found; using BASE_PARAMS")

preds, golds, pred_rows, missing_debug = build_preds(params, repair_mode=mode)
metric = metric_from_preds(preds, golds, name="v26_full_operational")
summary = {
    "version": "v26_full_next_session",
    "note": "Operational runner: freeze MC v25/v24.1, YNU params from v26 standard if available.",
    "params": params,
    "repair_mode": mode,
    "debug_loaded": DEBUG_PATH is not None,
    "option_preds_count": len(MC_DEBUG),
    "missing_debug_count": missing_debug,
    "metric": metric,
}
print("="*100)
print("V26 FULL OPERATIONAL")
print("="*100)
print(json.dumps({k: metric[k] for k in ["acc","macro_f1","weighted_f1","mc_macro_f1","ynu_macro_f1"]}, indent=2))
for lab in LABELS:
    d = metric["per_label"][lab]
    print(f"{lab:8s} F1={d['f1']:.3f} P={d['precision']:.3f} R={d['recall']:.3f} pred={d['pred_count']} gold={d['support']}")

save_json(OUT_DIR / "v26_full_summary.json", summary)
save_json(OUT_DIR / "v26_full_preds.json", pred_rows)
